# MUTCD Multimodal RAG (v3) — Colab / HPRC

Hierarchical hybrid retrieval (dense + sparse + knowledge-graph proximity +
rule-type weighting) over the MUTCD 11th Edition, with a VLM answer step
that can run either **locally on GPU** or via an **API** (Qwen3-VL-32B by
default through DashScope).

Run cells top to bottom, once per session.

## 0. First-cell setup (Colab) — skip this entire section on HPRC

In [ ]:
# COLAB ONLY — comment out this whole cell if running on HPRC.
import sys, os

# Mount Google Drive — this is where the PDF, figures, Qdrant snapshot, and
# HF model cache persist across sessions.
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

# CRITICAL: tell every subprocess (including ingest_v3.py launched via !python)
# that this is a Colab environment. sys.modules-based detection does NOT cross
# process boundaries, but environment variables DO.
os.environ["MRAG_ENV"] = "colab"

# Clone or update the repo
if not os.path.isdir("/content/MRAG_stp2"):
    !git clone https://github.com/hannanazad/MRAG_stp2.git /content/MRAG_stp2
else:
    !cd /content/MRAG_stp2 && git pull

sys.path.insert(0, "/content/MRAG_stp2")

# ───────────────────────────────────────────────────────────────────────────
# Step 1 — torch + torchvision matched to Colab's CUDA 12.4.
# Colab's base image ships torch 2.5.1, but transformers >= 4.51 includes a
# torch.load CVE check (CVE-2025-32434) that refuses to load .bin checkpoints
# on torch < 2.6. BGE-M3 (and most HF models that ship .bin) need torch 2.6+
# in order for sentence-transformers to load them.
!pip install -q --index-url https://download.pytorch.org/whl/cu124 \
    torch==2.6.0 torchvision==0.21.0

# Step 2 — the rest of the dependencies from requirements.txt
!pip install -q -r /content/MRAG_stp2/requirements.txt

# Step 3 — pip's bulk -r resolver silently skips enforcing CEILINGS when a
# satisfying lower-bound version is already installed. Colab's base image
# ships newer transformers / huggingface_hub / tokenizers / torchao than we
# can use, so we force them to the right versions explicitly:
#   - transformers   <4.55   avoids PreTrainedModel circular-import regression
#   - huggingface_hub <0.35  avoids the API-removal regression that broke
#                            transformers 4.54.1's runtime version check
#   - tokenizers     <0.22   matched to transformers 4.54.x
#   - torchao        <0.14   0.13 is the last release supporting torch 2.6
#                            (0.14+ uses torch.int1, which is torch 2.7+ only)
!pip install -q --no-deps --force-reinstall \
    "transformers>=4.49,<4.55" \
    "huggingface_hub>=0.34,<0.35" \
    "tokenizers>=0.21,<0.22" \
    "torchao>=0.13,<0.14"

# Step 4 — sanity-check the critical import path before moving on.
!python -c "from transformers import PreTrainedModel; print('transformers import OK:', PreTrainedModel.__name__)"


In [ ]:
# Paste your Qwen API key into Colab Secrets (key icon, left sidebar) under
# the name shown below, then run this cell. Never hard-code the key directly.
from google.colab import userdata
import os

SECRET_NAME = "QWEN" # <- the name you used in the Secrets panel

os.environ["VLM_API_KEY"] = userdata.get(SECRET_NAME)
print("API key set:", bool(os.environ.get("VLM_API_KEY")))


## 1. Environment sanity check (both Colab and HPRC)

In [ ]:
import os, sys
sys.path.insert(0, "/content/MRAG_stp2" if os.path.isdir("/content/MRAG_stp2") else ".")

from mrag.config import CFG

print("Environment :", CFG.environment)
print("Base dir    :", CFG.base_dir, "| exists:", CFG.base_dir.exists())
print("PDF path    :", CFG.pdf_path, "| exists:", CFG.pdf_path.exists())
print("Qdrant dir  :", CFG.qdrant_dir)
print("Cache dir   :", CFG.cache_dir)
print("HF cache    :", CFG.hf_home)
print("VLM provider:", CFG.vlm_provider)
print("VLM model   :", CFG.vlm_model_api if CFG.vlm_provider == "api" else CFG.vlm_model)
print("API key set :", bool(os.environ.get(CFG.api_key_env_var)))

assert CFG.pdf_path.exists(), (
    f"No PDF found at {CFG.pdf_path} or anywhere in {CFG.base_dir}. "
    f"Upload the MUTCD PDF to {CFG.base_dir} before continuing."
)


## 1.5 Recover or build the Qdrant vector store

If you've run ingestion successfully before, this restores the existing
Drive snapshot into the fast local Colab disk in seconds. If no snapshot
exists yet, it falls through to running full ingestion (first time only,
~30–45 min; subsequent reruns are much faster since chunks/figures/KG are
cached and only re-embedding + upsert happens, ~5 min).

In [ ]:
import shutil, tarfile
from pathlib import Path
from qdrant_client import QdrantClient

local_qdrant = CFG.qdrant_dir                    # /content/qdrant_db (fast local SSD)
drive_qdrant_tar = CFG.base_dir / "qdrant_db.tar"  # snapshot on Drive

def _qdrant_ok(path: Path) -> bool:
    if not path.exists():
        return False
    try:
        c = QdrantClient(path=str(path))
        try:
            names = {col.name for col in c.get_collections().collections}
            return CFG.coll_chunks in names
        finally:
            c.close()
    except Exception:
        return False

if _qdrant_ok(local_qdrant):
    print(f"Qdrant already populated at {local_qdrant}. Nothing to do.")

elif drive_qdrant_tar.exists():
    print(f"Found Qdrant snapshot on Drive at {drive_qdrant_tar} ({drive_qdrant_tar.stat().st_size/1e6:.0f} MB).")
    print(f"Extracting to local disk at {local_qdrant} ...")
    shutil.rmtree(local_qdrant, ignore_errors=True)
    local_qdrant.mkdir(parents=True, exist_ok=True)
    with tarfile.open(drive_qdrant_tar, "r") as tar:
        tar.extractall(local_qdrant.parent)
    assert _qdrant_ok(local_qdrant), "Extraction completed but collection not found — investigate."
    print("Local Qdrant verified OK.")

else:
    print("No usable Qdrant store found anywhere. Running ingestion.")
    if CFG.graph_pickle.exists():
        print("KG exists but Qdrant is empty — running ingestion. Cached steps will")
        print("be skipped automatically; expect ~5-10 min for embeddings + upsert.")
    else:
        print("First-ever ingestion — this takes ~30-45 min on an A100.")
    !cd /content/MRAG_stp2 && python scripts/ingest_v3.py
    assert _qdrant_ok(local_qdrant), "Ingestion finished but Qdrant still empty — check the log above for errors."
    print("Ingestion verified OK.")


## 1.6 Snapshot Qdrant back to Drive (Colab only)

Run this after any successful ingestion so the *next* session can restore
instantly via the recovery cell above, instead of re-running ingestion.

In [ ]:
if CFG.environment == "colab":
    local_qdrant = CFG.qdrant_dir  # define here so this cell can run independently
    drive_qdrant_tar = CFG.base_dir / "qdrant_db.tar"
    
    if local_qdrant.exists():
        print(f"Snapshotting {local_qdrant} -> {drive_qdrant_tar} ...")
        drive_qdrant_tar.parent.mkdir(parents=True, exist_ok=True)
        with tarfile.open(drive_qdrant_tar, "w") as tar:
            tar.add(local_qdrant, arcname=local_qdrant.name)
        print(f"Snapshot complete ({drive_qdrant_tar.stat().st_size/1e6:.0f} MB).")
    else:
        print("No local Qdrant to snapshot.")


## 2. Initialize the pipeline (one time per session, ~30–60 s in API mode)

In [ ]:
import logging, os
logging.basicConfig(level=logging.INFO, format='%(name)s - %(message)s')

# Guard: if API mode is active but no key is set, fail with a clear message
if CFG.vlm_provider == "api" and not os.environ.get(CFG.api_key_env_var):
    raise RuntimeError(
        "CFG.vlm_provider is 'api' but no API key is set.\n"
        "Either:\n"
        "  1. Run cell 0.5 to load the key from Colab Secrets, or\n"
        "  2. Edit mrag/config.py and set vlm_provider='local' to use the local 7B model."
    )

from mrag.ask import init_pipeline
pipeline = init_pipeline()
print("VLM loaded :", pipeline.vlm.loaded_name if pipeline.vlm else "none")
print("KG         :", pipeline.kg.g.number_of_nodes(), "nodes,", pipeline.kg.g.number_of_edges(), "edges")


## 3. Ask

In [ ]:
from mrag.ask import ask

_ = ask("What is required when installing a STOP sign at an all-way stop intersection?")


In [ ]:
_ = ask("Explain Figure 2B-1 and the plaques it shows", show_scores=True)


In [ ]:
_ = ask("What does MUTCD say about pedestrian hybrid beacons?", show_text=True)


## 4. Inspect the knowledge graph

In [ ]:
kg = pipeline.kg
print(kg.g.number_of_nodes(), "nodes,", kg.g.number_of_edges(), "edges")


In [ ]:
# Neighbourhood of a sign code
# (fill in a real sign code node id from your graph, e.g. "signcode:R1-1")


In [ ]:
# Which figures does Section 2B.04 directly link to via cross-references in its chunks?


## 5. Debug retrieval (no VLM call)

In [ ]:
from mrag.retrieval import Retriever
